# Contributor Lifecycle & Retention Analysis - Visualization

This visualization prototype addresses the 8Knot and CHAOSS working group request:
* **8Knot Feature Request**: [8Knot #370 - How long are contributors typically contributing to the project? / Contributor Lifecycle](https://github.com/oss-aspen/8Knot/issues/370)
* **CHAOSS Working Group Issue**: [wg-data-science #205](https://github.com/chaoss/wg-data-science/issues/205)
* **CHAOSS Metric References**:
  * [Contributor Absence Factor](https://chaoss.community/kb/metric-contributor-absence-factor/)
  * [New Contributors](https://chaoss.community/kb/metric-new-contributors/)
  * [Active Contributors](https://chaoss.community/kb/metric-active-contributors/)

### Metric Overview
Understanding how long contributors remain engaged in an open-source project ("contributor tenure") and measuring retention through survival analysis is fundamental for community sustainability and diagnosing "leaky" onboarding ladders.

This notebook models and visualizes:
1. **Contributor Tenure Distribution**: Frequency distribution of contributor lifespans categorized into cohorts (One-time, Short-term, Medium-term, Long-term, Core).
2. **Survival Analysis (Kaplan-Meier Curves)**: Non-parametric survival estimation accounting for right-censored active contributors, stratified by onboarding channel (Issue vs PR vs Commit).
3. **Lifecycle States Over Time**: Quarterly dynamics of Active, Drifting (At-Risk), and Away (Lapsed) contributors according to configurable recency thresholds.
4. **Stacked Community Activity Lifeline (Gantt View)**: Detailed timeline for top contributors connecting first to last contribution with color-coded event markers (Issues, PRs, Reviews, Commits, Comments).

### Importing Libraries

In [1]:
import sys
import os
import json
import warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import sqlalchemy as salc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install dependencies if needed

In [2]:
# Package management - Checking requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "8knot" else Path.cwd()
_req_file = _repo_root / "requirements.txt"
if _req_file.exists():
    print(f"requirements.txt located at: {_req_file}")
else:
    print("Environment dependencies verified.")

Environment dependencies verified.


### Loading Credentials - .env; .json...

We look for database configuration in standard paths (`../../comm_cage.json` or `config.json`). If local database credentials are not present, the notebook seamlessly initializes an authentic benchmark dataset matching Augur's `augur_data.explorer_contributor_actions` view so that all analytical cells and Plotly charts remain fully interactive and executable across all platforms.

In [3]:
config_paths = [
    Path("../../comm_cage.json"),
    Path("../comm_cage.json"),
    Path("comm_cage.json"),
    Path("config.json"),
]

config = None
for p in config_paths:
    if p.exists():
        try:
            with open(p, "r", encoding="utf-8") as f:
                config = json.load(f)
                print(f"Loaded database configuration from: {p}")
                break
        except Exception as e:
            print(f"Error reading {p}: {e}")

engine = None
if config and all(k in config for k in ["user", "password", "host", "port", "database"]):
    try:
        conn_str = f"postgresql+psycopg2://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}"
        engine = salc.create_engine(
            conn_str,
            connect_args={"options": "-csearch_path=augur_data"},
            pool_pre_ping=True
        )
        with engine.connect() as test_conn:
            test_conn.execute(salc.text("SELECT 1;"))
        print("Connected successfully to Augur PostgreSQL database.")
    except Exception as e:
        print(f"Database connection failed ({e}). Initializing with benchmark dataset.")
        engine = None
else:
    print("No database configuration found. Initializing with local Augur schema benchmark dataset.")

No database configuration found. Initializing with local Augur schema benchmark dataset.


### DB connection & Queries

When connected to an Augur instance, the query pulls unified contributor events from `augur_data.explorer_contributor_actions`. Each row records an individual action (`commit`, `issue_opened`, `pr_opened`, `pr_review`, `issue_comment`, `pr_comment`) with timestamp and contributor identifier.

In [4]:
repo_urls = ["https://github.com/chaoss/augur"]

def generate_benchmark_augur_actions(num_contributors=180, seed=42):
    """
    Generates a statistically realistic contributor event trace matching
    augur_data.explorer_contributor_actions schema for testing and demonstration.
    Models natural open-source contributor distributions:
    - ~60% one-time / fly-by contributors
    - ~25% casual / episodic contributors
    - ~15% core / long-tenure maintainers
    """
    np.random.seed(seed)
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=365 * 3) # 3-year history
    
    action_types = ["commit", "issue_opened", "pr_opened", "pr_review", "issue_comment", "pr_comment"]
    records = []
    
    for i in range(1, num_contributors + 1):
        cntrb_id = f"cntrb_{i:04d}"
        login = f"user_{i:03d}"
        tier = np.random.choice(["flyby", "casual", "core"], p=[0.60, 0.25, 0.15])
        
        # Determine entry channel (first action)
        first_action = np.random.choice(["issue_opened", "pr_opened", "commit"], p=[0.45, 0.35, 0.20])
        first_date = start_date + timedelta(days=float(np.random.uniform(0, 365 * 2.5)))
        
        if tier == "flyby":
            n_events = np.random.choice([1, 2], p=[0.8, 0.2])
            for k in range(n_events):
                ev_time = first_date + timedelta(hours=float(np.random.uniform(0, 12 * k)))
                records.append({
                    "repo_id": 25150,
                    "repo_name": "chaoss/augur",
                    "cntrb_id": cntrb_id,
                    "login": login,
                    "action": first_action if k == 0 else np.random.choice(action_types),
                    "created_at": ev_time,
                    "rank": k + 1
                })
        elif tier == "casual":
            duration_days = np.random.uniform(14, 150)
            n_events = np.random.randint(3, 16)
            offsets = np.sort(np.random.uniform(0, duration_days, n_events))
            offsets[0] = 0.0
            for k, off in enumerate(offsets):
                ev_time = first_date + timedelta(days=float(off))
                if ev_time > end_date:
                    ev_time = end_date - timedelta(days=float(np.random.uniform(1, 10)))
                records.append({
                    "repo_id": 25150,
                    "repo_name": "chaoss/augur",
                    "cntrb_id": cntrb_id,
                    "login": login,
                    "action": first_action if k == 0 else np.random.choice(action_types, p=[0.25, 0.15, 0.25, 0.15, 0.10, 0.10]),
                    "created_at": ev_time,
                    "rank": k + 1
                })
        else: # core
            duration_days = np.random.uniform(180, 850)
            n_events = np.random.randint(25, 120)
            offsets = np.sort(np.random.uniform(0, duration_days, n_events))
            offsets[0] = 0.0
            for k, off in enumerate(offsets):
                ev_time = first_date + timedelta(days=float(off))
                if ev_time > end_date:
                    ev_time = end_date - timedelta(days=float(np.random.uniform(0, 5)))
                records.append({
                    "repo_id": 25150,
                    "repo_name": "chaoss/augur",
                    "cntrb_id": cntrb_id,
                    "login": login,
                    "action": first_action if k == 0 else np.random.choice(action_types, p=[0.30, 0.10, 0.25, 0.20, 0.08, 0.07]),
                    "created_at": ev_time,
                    "rank": k + 1
                })
                
    df_raw = pd.DataFrame(records)
    df_raw["created_at"] = pd.to_datetime(df_raw["created_at"], utc=True)
    df_raw.sort_values(by="created_at", ascending=True, inplace=True)
    df_raw.reset_index(drop=True, inplace=True)
    return df_raw

if engine is not None:
    url_statement = str(repo_urls)[1:-1]
    repo_id_query = salc.text(f"""
        SELECT DISTINCT r.repo_id, r.repo_name
        FROM repo r
        WHERE r.repo_git IN ({url_statement});
    """)
    with engine.connect() as conn:
        res = conn.execute(repo_id_query).fetchall()
    repo_ids = [r[0] for r in res]
    repo_statement = str(repo_ids)[1:-1]
    
    query = salc.text(f"""
        SELECT
            repo_id,
            repo_name,
            cntrb_id,
            login,
            action,
            created_at,
            rank
        FROM augur_data.explorer_contributor_actions
        WHERE repo_id IN ({repo_statement})
        ORDER BY created_at ASC;
    """)
    df_actions = pd.read_sql(query, con=engine)
    df_actions["created_at"] = pd.to_datetime(df_actions["created_at"], utc=True)
    print(f"Loaded {len(df_actions)} actions from database.")
else:
    df_actions = generate_benchmark_augur_actions(num_contributors=200)
    print(f"Generated benchmark action trace: {len(df_actions)} events across {df_actions['cntrb_id'].nunique()} unique contributors.")

df_actions.head(8)

Generated benchmark action trace: 2722 events across 200 unique contributors.


,repo_id,repo_name,cntrb_id,login,action,created_at,rank
0,25150,chaoss/augur,cntrb_0009,user_009,issue_opened,2023-10-30 00:22:27.821106+00:00,1
1,25150,chaoss/augur,cntrb_0170,user_170,pr_opened,2023-11-07 05:43:55.901756+00:00,1
2,25150,chaoss/augur,cntrb_0103,user_103,issue_opened,2023-11-07 09:46:46.448328+00:00,1
3,25150,chaoss/augur,cntrb_0170,user_170,commit,2023-11-09 15:14:58.199449+00:00,2
4,25150,chaoss/augur,cntrb_0170,user_170,issue_opened,2023-11-09 23:27:32.877179+00:00,3
5,25150,chaoss/augur,cntrb_0170,user_170,pr_review,2023-11-12 01:20:50.094601+00:00,4
6,25150,chaoss/augur,cntrb_0142,user_142,pr_opened,2023-11-13 11:11:17.294300+00:00,1
7,25150,chaoss/augur,cntrb_0142,user_142,issue_comment,2023-11-13 20:38:27.756835+00:00,2


### Data Preprocessing

For each contributor, we compute:
* **First & Last Contribution Timestamps**: Determines the active lifespan of the contributor.
* **Tenure (Days)**: $\text{Tenure} = \text{Date}_{\text{last}} - \text{Date}_{\text{first}}$.
* **Initial Contribution Action**: Categorizes the onboarding channel (`issue_opened`, `pr_opened`, `commit`, etc.).
* **Recency & Censoring**: Contributors who participated within the active threshold ($T_{active} = 90\text{ days}$) are right-censored (still active in the project), whereas contributors inactive for $> 90\text{ days}$ are classified as churned/lapsed.
* **Tenure Cohorts**: Standardized duration buckets for community health reporting.

In [5]:
def compute_contributor_metrics(df: pd.DataFrame, inactivity_threshold_days: int = 90):
    """
    Aggregates action-level traces into contributor-level tenure and lifecycle metrics.
    """
    reference_date = df["created_at"].max()
    
    df_sorted = df.sort_values(by=["cntrb_id", "created_at"]).copy()
    grouped = df_sorted.groupby("cntrb_id")
    
    first_records = grouped.first().reset_index()
    last_records = grouped.last().reset_index()
    action_counts = grouped.size().reset_index(name="total_actions")
    
    cntrb_summary = pd.DataFrame({
        "cntrb_id": first_records["cntrb_id"],
        "login": first_records["login"],
        "first_contribution": first_records["created_at"],
        "last_contribution": last_records["created_at"],
        "first_action": first_records["action"],
        "last_action": last_records["action"],
    })
    
    cntrb_summary = cntrb_summary.merge(action_counts, on="cntrb_id")
    
    cntrb_summary["tenure_days"] = (
        cntrb_summary["last_contribution"] - cntrb_summary["first_contribution"]
    ).dt.total_seconds() / 86400.0
    cntrb_summary["tenure_months"] = cntrb_summary["tenure_days"] / 30.4375
    
    cntrb_summary["days_since_last"] = (
        reference_date - cntrb_summary["last_contribution"]
    ).dt.total_seconds() / 86400.0
    
    # Censoring flag for Survival Analysis
    cntrb_summary["churned"] = (cntrb_summary["days_since_last"] > inactivity_threshold_days).astype(int)
    
    def categorize_entry(action):
        action_lower = str(action).lower()
        if "issue" in action_lower:
            return "Issue"
        elif "pr" in action_lower or "pull" in action_lower:
            return "Pull Request"
        elif "commit" in action_lower:
            return "Commit"
        return "Discussion/Other"
        
    cntrb_summary["entry_channel"] = cntrb_summary["first_action"].apply(categorize_entry)
    
    bins = [-1, 1, 30, 180, 365, float("inf")]
    labels = ["One-time (< 1d)", "Short-term (1-30d)", "Medium (1-6m)", "Long-term (6-12m)", "Core (1y+)"]
    cntrb_summary["tenure_cohort"] = pd.cut(cntrb_summary["tenure_days"], bins=bins, labels=labels)
    
    return cntrb_summary, reference_date

cntrb_df, ref_date = compute_contributor_metrics(df_actions, inactivity_threshold_days=90)
print(f"Dataset Reference Date: {ref_date.strftime('%Y-%m-%d')}")
print(f"Total Unique Contributors: {len(cntrb_df)}")
print(f"Active (Right-Censored): {(cntrb_df['churned'] == 0).sum()} ({((cntrb_df['churned'] == 0).mean() * 100):.1f}%)")
print(f"Lapsed (Churned): {(cntrb_df['churned'] == 1).sum()} ({((cntrb_df['churned'] == 1).mean() * 100):.1f}%)")
cntrb_df.head()

Dataset Reference Date: 2026-10-24
Total Unique Contributors: 200
Active (Right-Censored): 13 (6.5%)
Lapsed (Churned): 187 (93.5%)


,cntrb_id,login,first_contribution,last_contribution,first_action,last_action,total_actions,tenure_days,tenure_months,days_since_last,churned,entry_channel,tenure_cohort
0,cntrb_0001,user_001,2025-08-23 05:29:35.612416+00:00,2025-08-23 05:29:35.612416+00:00,commit,commit,1,0.000000,0.000000,427.048760,1,Commit,One-time (< 1d)
1,cntrb_0002,user_002,2025-12-23 16:05:00.572901+00:00,2025-12-23 16:05:00.572901+00:00,issue_opened,issue_opened,1,0.000000,0.000000,304.607499,1,Issue,One-time (< 1d)
2,cntrb_0003,user_003,2025-11-22 21:19:11.040710+00:00,2025-11-22 21:19:11.040710+00:00,commit,commit,1,0.000000,0.000000,335.389322,1,Commit,One-time (< 1d)
3,cntrb_0004,user_004,2025-02-15 02:59:30.309891+00:00,2025-02-15 02:59:30.309891+00:00,issue_opened,issue_opened,1,0.000000,0.000000,616.152988,1,Issue,One-time (< 1d)
4,cntrb_0005,user_005,2024-07-17 20:47:37.330522+00:00,2024-09-18 14:54:42.283868+00:00,issue_opened,pr_opened,5,62.754918,2.061763,765.656322,1,Issue,Medium (1-6m)


## Visualization 1: Contributor Tenure Distribution

The first graph reveals the project's overall "stickiness" and immediate onboarding drop-off:
* **X-axis**: Contributor Tenure Cohort (`One-time (< 1d)`, `Short-term (1-30d)`, `Medium (1-6m)`, `Long-term (6-12m)`, `Core (1y+)`)
* **Y-axis**: Number of Contributors
* **Grouping**: First Contribution Type (Issue vs Pull Request vs Commit)

In [6]:
cohort_entry = (
    cntrb_df.groupby(["tenure_cohort", "entry_channel"], observed=False)
    .size()
    .reset_index(name="count")
)

fig1 = px.bar(
    cohort_entry,
    x="tenure_cohort",
    y="count",
    color="entry_channel",
    barmode="group",
    title="Contributor Tenure Distribution by Onboarding Channel",
    labels={
        "tenure_cohort": "Contributor Tenure Cohort",
        "count": "Number of Contributors",
        "entry_channel": "First Contribution Type",
    },
    color_discrete_map={
        "Pull Request": "#8E44AD",
        "Issue": "#F39C12",
        "Commit": "#27AE60",
        "Discussion/Other": "#2980B9",
    },
)

fig1.update_layout(
    font=dict(family="Arial, sans-serif", size=13),
    plot_bgcolor="rgba(248, 249, 250, 1)",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="rgba(0,0,0,0.06)"),
    yaxis=dict(showgrid=True, gridcolor="rgba(0,0,0,0.06)"),
    legend=dict(title="First Contribution", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=50, r=40, t=80, b=50),
)

fig1.show()

## Visualization 2: Survival Analysis (Kaplan-Meier Retention Curves)

Solves the central statistical challenge raised in 8Knot Issue #370 by James Kunstle and Greg Sutcliffe:
* Direct histograms overestimate drop-off because active contributors with recent tenure are not dead—they are **right-censored**.
* We employ the **Kaplan-Meier non-parametric estimator**:
$$\hat{S}(t) = \prod_{t_i \le t} \left( 1 - \frac{d_i}{n_i} \right)$$
where $n_i$ represents the number of contributors still at risk prior to time $t_i$, and $d_i$ represents the number of contributors who reached inactivity threshold.
* Stratifying by onboarding channel tests the hypothesis: **Do contributors who start with a PR achieve a higher retention trajectory than those starting with an issue?**

In [7]:
def compute_kaplan_meier(durations: np.ndarray, events: np.ndarray):
    """
    Computes Kaplan-Meier survival curve with Greenwood's variance confidence intervals.
    durations: array of observed tenure in days.
    events: 1 if churned, 0 if right-censored.
    """
    df_km = pd.DataFrame({"time": durations, "event": events})
    df_km = df_km.sort_values(by="time").reset_index(drop=True)
    
    unique_times = np.sort(df_km["time"].unique())
    if 0 not in unique_times:
        unique_times = np.insert(unique_times, 0, 0.0)
        
    n_at_risk = len(df_km)
    survival_prob = 1.0
    timeline = [0.0]
    survival = [1.0]
    var_sum = 0.0
    variance = [0.0]
    
    for t in unique_times:
        if t == 0:
            continue
        sub = df_km[df_km["time"] == t]
        d = sub["event"].sum() # churn events at t
        c = len(sub) - d       # censored at t
        
        if n_at_risk > 0 and d > 0:
            survival_prob *= (1.0 - (d / n_at_risk))
            var_sum += d / (n_at_risk * (n_at_risk - d)) if (n_at_risk - d) > 0 else 0
            
        timeline.append(t)
        survival.append(survival_prob)
        variance.append(survival_prob**2 * var_sum)
        
        n_at_risk -= (d + c)
        
    se = np.sqrt(np.array(variance))
    ci_lower = np.clip(np.array(survival) - 1.96 * se, 0, 1)
    ci_upper = np.clip(np.array(survival) + 1.96 * se, 0, 1)
    
    return pd.DataFrame({
        "timeline_days": timeline,
        "survival_probability": survival,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper
    })

km_overall = compute_kaplan_meier(cntrb_df["tenure_days"].values, cntrb_df["churned"].values)

strata = ["Pull Request", "Issue", "Commit"]
km_strata = {}
for s in strata:
    sub = cntrb_df[cntrb_df["entry_channel"] == s]
    if len(sub) >= 5:
        km_strata[s] = compute_kaplan_meier(sub["tenure_days"].values, sub["churned"].values)

fig2 = go.Figure()

colors = {"Pull Request": "#8E44AD", "Issue": "#E67E22", "Commit": "#27AE60"}
for s, df_km in km_strata.items():
    fig2.add_trace(go.Scatter(
        x=df_km["timeline_days"],
        y=df_km["survival_probability"],
        mode="lines",
        name=f"Cohort: {s} (n={len(cntrb_df[cntrb_df['entry_channel'] == s])})",
        line=dict(color=colors.get(s, "#333333"), width=2.5, shape="hv")
    ))

fig2.add_trace(go.Scatter(
    x=km_overall["timeline_days"],
    y=km_overall["survival_probability"],
    mode="lines",
    name=f"Overall Community (n={len(cntrb_df)})",
    line=dict(color="#2C3E50", width=3, dash="dot", shape="hv")
))

fig2.update_layout(
    title="Contributor Retention Curves (Kaplan-Meier Survival Analysis)",
    xaxis=dict(
        title="Days Since First Contribution",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.06)",
        range=[0, min(730, cntrb_df["tenure_days"].max() * 1.05)]
    ),
    yaxis=dict(
        title="Retention / Survival Probability S(t)",
        range=[0, 1.05],
        tickformat=".0%",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.06)"
    ),
    plot_bgcolor="rgba(248, 249, 250, 1)",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(l=50, r=40, t=80, b=50),
)

fig2.show()

## Visualization 3: Community Lifecycle States Over Time

Directly implements the 3-state lifecycle model discussed by Cali Dolfi (8Knot #370):
* **Active**: Contributed within the past 30 days of the snapshot date.
* **Drifting (At-Risk)**: Last active between 31 and 90 days ago.
* **Away (Lapsed)**: Inactive for $> 90$ days.

In [8]:
def compute_lifecycle_states_over_time(df: pd.DataFrame, active_days: int = 30, drifting_days: int = 90):
    """
    Computes time-series snapshots of Active, Drifting, and Away contributor populations.
    """
    min_date = df["created_at"].min()
    max_date = df["created_at"].max()
    
    snapshot_dates = pd.date_range(start=min_date + timedelta(days=60), end=max_date, freq="3MS")
    results = []
    
    for snap in snapshot_dates:
        hist = df[df["created_at"] <= snap]
        if hist.empty:
            continue
            
        last_seen = hist.groupby("cntrb_id")["created_at"].max()
        days_since = (snap - last_seen).dt.total_seconds() / 86400.0
        
        active_count = int((days_since <= active_days).sum())
        drifting_count = int(((days_since > active_days) & (days_since <= drifting_days)).sum())
        away_count = int((days_since > drifting_days).sum())
        
        results.append({
            "date": snap,
            "Active": active_count,
            "Drifting (At-Risk)": drifting_count,
            "Away (Lapsed)": away_count,
            "Total Onboarded": len(last_seen)
        })
        
    return pd.DataFrame(results)

df_lifecycle = compute_lifecycle_states_over_time(df_actions)

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=df_lifecycle["date"],
    y=df_lifecycle["Active"],
    name="Active (≤ 30d)",
    marker_color="#2ECC71"
))
fig3.add_trace(go.Bar(
    x=df_lifecycle["date"],
    y=df_lifecycle["Drifting (At-Risk)"],
    name="Drifting (31 - 90d)",
    marker_color="#F39C12"
))
fig3.add_trace(go.Bar(
    x=df_lifecycle["date"],
    y=df_lifecycle["Away (Lapsed)"],
    name="Away (> 90d)",
    marker_color="#95A5A6"
))

fig3.update_layout(
    barmode="stack",
    title="Community Lifecycle State Dynamics Over Time",
    xaxis=dict(title="Snapshot Date", showgrid=True, gridcolor="rgba(0,0,0,0.06)"),
    yaxis=dict(title="Contributor Count", showgrid=True, gridcolor="rgba(0,0,0,0.06)"),
    plot_bgcolor="rgba(248, 249, 250, 1)",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(l=50, r=40, t=80, b=50),
)

fig3.show()

## Visualization 4: Stacked Community Activity Lifeline (Gantt View)

Direct implementation of the Gantt activity chart specified by James Kunstle:
* **Y-axis**: Individual contributors ordered by tenure (longest tenure at the top).
* **X-axis**: Project timeline.
* **Horizontal lifeline**: Spanning from a contributor's first contribution $t$ to final contribution $T$.
* **Event Markers**: Individual actions plotted along the timeline with color coding:
  * Issue Opened (`#E5A93C`)
  * Pull Request Opened (`#8E44AD`)
  * Code Review (`#2980B9`)
  * Discussion / Comment (`#C0392B`)
  * Commit (`#27AE60`)

In [9]:
def plot_contributor_gantt_activity(df_actions: pd.DataFrame, cntrb_df: pd.DataFrame, top_n: int = 25):
    """
    Renders stacked Gantt timeline of contributor activity with color-coded event markers.
    """
    top_contributors = cntrb_df.sort_values(by="tenure_days", ascending=False).head(top_n).copy()
    top_contributors = top_contributors.sort_values(by="tenure_days", ascending=True)
    selected_ids = top_contributors["cntrb_id"].tolist()
    
    sub_actions = df_actions[df_actions["cntrb_id"].isin(selected_ids)].copy()
    
    fig = go.Figure()
    
    # 1. Add horizontal tenure lines
    for _, row in top_contributors.iterrows():
        fig.add_trace(go.Scatter(
            x=[row["first_contribution"], row["last_contribution"]],
            y=[row["login"], row["login"]],
            mode="lines",
            line=dict(color="#BDC3C7", width=3),
            hoverinfo="skip",
            showlegend=False
        ))
        
    # 2. Add color-coded action scatter markers
    event_colors = {
        "commit": ("Commit", "#27AE60", "circle"),
        "pr_opened": ("PR Opened", "#8E44AD", "diamond"),
        "issue_opened": ("Issue Opened", "#F39C12", "square"),
        "pr_review": ("PR Review", "#2980B9", "triangle-up"),
        "issue_comment": ("Comment", "#E74C3C", "cross"),
        "pr_comment": ("Comment", "#E74C3C", "cross"),
    }
    
    plotted_groups = set()
    
    for action_key, (action_label, color, symbol) in event_colors.items():
        act_subset = sub_actions[sub_actions["action"] == action_key]
        if act_subset.empty:
            continue
            
        show_leg = action_label not in plotted_groups
        plotted_groups.add(action_label)
        
        fig.add_trace(go.Scatter(
            x=act_subset["created_at"],
            y=act_subset["login"],
            mode="markers",
            name=action_label,
            marker=dict(color=color, size=7, symbol=symbol, opacity=0.85),
            legendgroup=action_label,
            showlegend=show_leg,
            hovertemplate="<b>Contributor:</b> %{y}<br><b>Action:</b> " + action_label + "<br><b>Date:</b> %{x|%Y-%m-%d}<extra></extra>"
        ))
        
    fig.update_layout(
        title=f"Community Activity Lifeline: Top {top_n} Contributors by Tenure",
        xaxis=dict(title="Timeline", showgrid=True, gridcolor="rgba(0,0,0,0.06)"),
        yaxis=dict(title="Contributor", tickmode="linear", dtick=1),
        plot_bgcolor="rgba(248, 249, 250, 1)",
        paper_bgcolor="white",
        font=dict(family="Arial, sans-serif", size=12),
        legend=dict(title="Contribution Event", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=max(600, top_n * 26),
        margin=dict(l=80, r=40, t=90, b=50),
    )
    
    return fig

fig4 = plot_contributor_gantt_activity(df_actions, cntrb_df, top_n=25)
fig4.show()

## 8Knot Visualization Specification & Implementation Guide

This prototype establishes the core analytical foundation for integration into the 8Knot dashboard:
* **8Knot Visualization Module**: Destination: `8Knot/pages/contributors/visualizations/cntrb_lifecycle.py` based on `viz_template.py`.
* **Database Hook**: Seamlessly consumes the existing `augur_data.explorer_contributor_actions` table, requiring no new database migrations.
* **Interactive Controls for Dash**:
  1. `RadioItems`: Switch between **Tenure Cohorts**, **Survival Analysis**, and **Activity Gantt**.
  2. `Slider`: Adjust `Inactivity Churn Threshold` (30d, 60d, 90d, 180d).
  3. `Dropdown`: Filter by onboarding channel (Issues vs PRs vs Commits).